# PH1050 Project 8 — Part 3: kNN classifier (metal cylinder vs rock)
Pipeline: `StandardScaler` -> kNN. k is chosen by 5-fold `StratifiedKFold` CV and compared with the majority-class baseline.


In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Project8
except ImportError:
    print('Not in Colab - using the current folder')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, train_test_split

SEED = 42
K_GRID = list(range(1, 32, 2))   # odd k avoids voting ties
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## 1. Load the datasets

In [ ]:
def get_xy(df):
    feats = [c for c in df.columns if c.startswith('energy_bin_')]
    metal = (df['class'] == 'metal_cylinder') if 'class' in df.columns else (df['label'] == 'M')
    return df[feats].to_numpy(), metal.astype(int).to_numpy()

def add_noise(df, sigma, seed=2026):
    # same recipe as the data-collection notebook: extra Gaussian noise, clipped to [0,1]
    feats = [c for c in df.columns if c.startswith('energy_bin_')]
    out = df.copy(); rng = np.random.default_rng(seed)
    out[feats] = np.clip(out[feats].to_numpy() + rng.normal(0, sigma, (len(out), len(feats))), 0, 1)
    return out

syn = pd.read_csv('synthetic_echo_returns_400.csv')
datasets = {'synthetic': syn, 'synthetic + noise 0.10': add_noise(syn, 0.10)}

# Real data (second dataset): cleaned file if present, else download from UCI.
try:
    if os.path.exists('uci_sonar_clean.csv'):
        uci = pd.read_csv('uci_sonar_clean.csv')
    else:
        !pip -q install ucimlrepo
        from ucimlrepo import fetch_ucirepo
        s = fetch_ucirepo(id=151)
        uci = s.data.features.copy()
        uci.columns = [f'energy_bin_{i:02d}' for i in range(1, uci.shape[1] + 1)]
        uci['label'] = s.data.targets.iloc[:, 0].astype(str).str.strip().str.upper().values
    if 'class_name' in uci.columns: uci = uci.rename(columns={'class_name': 'class'})
    datasets['UCI Sonar (real)'] = uci
except Exception as e:
    print('UCI Sonar not loaded:', e)

for n, d in datasets.items():
    X, y = get_xy(d)
    print(f'{n:28s} {X.shape}  metal={y.sum()}  rock={len(y)-y.sum()}')

## 2. Choose k by 5-fold CV and compare with the baseline
For each k we get 5 fold accuracies and report mean ± std (sample std). *Nested* CV picks k inside each training fold only, so its accuracy is not biased by choosing k on the same folds.

In [ ]:
def knn(k): return Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=k))])

def run(X, y):
    base = cross_val_score(DummyClassifier(strategy='most_frequent'), X, y, cv=cv)
    mean = np.array([cross_val_score(knn(k), X, y, cv=cv).mean() for k in K_GRID])
    std  = np.array([cross_val_score(knn(k), X, y, cv=cv).std(ddof=1) for k in K_GRID])
    ib = int(np.where(mean == mean.max())[0][-1])          # ties -> larger k (smoother)
    inner = StratifiedKFold(5, shuffle=True, random_state=SEED + 1)
    nested = cross_val_score(GridSearchCV(knn(5), {'knn__n_neighbors': K_GRID}, cv=inner), X, y, cv=cv)
    return dict(base=(base.mean(), base.std(ddof=1)), mean=mean, std=std, best_k=K_GRID[ib],
                best=(mean[ib], std[ib]), nested=(nested.mean(), nested.std(ddof=1)))

results = {n: run(*get_xy(d)) for n, d in datasets.items()}
f = lambda t: f'{t[0]:.3f} ± {t[1]:.3f}'
summary = pd.DataFrame([{'dataset': n, 'baseline acc': f(r['base']), 'best k': r['best_k'],
    'CV acc at best k': f(r['best']), 'nested CV acc': f(r['nested'])} for n, r in results.items()])
summary

In [ ]:
plt.figure(figsize=(8, 5))
for n, r in results.items():
    plt.errorbar(K_GRID, r['mean'], yerr=r['std'], marker='o', capsize=2, label=n)
plt.axhline(0.5, ls='--', c='gray', label='majority baseline (balanced classes)')
plt.xlabel('k (number of neighbours)'); plt.ylabel('5-fold CV accuracy'); plt.title('kNN: accuracy vs k')
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig('knn_accuracy_vs_k.png', dpi=160); plt.show()

## 3. Held-out report on one dataset
Fit kNN with the CV-chosen k on an 80/20 stratified split and print the classification report (precision / recall per class). A 20% split of the UCI data is only ~42 samples, so treat this as an illustration; the CV numbers above are the result to quote.

In [ ]:
name = 'UCI Sonar (real)' if 'UCI Sonar (real)' in datasets else 'synthetic'
X, y = get_xy(datasets[name])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
k = results[name]['best_k']
model = knn(k).fit(Xtr, ytr); pred = model.predict(Xte)
print(f'{name}: k = {k}, test accuracy {model.score(Xte, yte):.3f} on {len(yte)} samples\n')
print(classification_report(yte, pred, target_names=['rock', 'metal']))
print('Confusion matrix (rows = true rock/metal, cols = predicted):'); print(confusion_matrix(yte, pred))

## 4. Noise sweep on the synthetic data
At low noise the classes separate perfectly, so many k tie at 100% and the 'best k' means nothing. More noise shows how the best k responds.

In [ ]:
rows = []
for s in [0, 0.1, 0.2, 0.4, 0.6, 0.8]:
    X, y = get_xy(syn if s == 0 else add_noise(syn, s))
    sc = [cross_val_score(knn(k), X, y, cv=cv) for k in K_GRID]
    m = np.array([a.mean() for a in sc]); sd = np.array([a.std(ddof=1) for a in sc])
    ib = int(np.where(m == m.max())[0][-1])
    rows.append(dict(extra_noise=s, best_k=K_GRID[ib], cv_acc=round(m[ib], 3), cv_std=round(sd[ib], 3),
                     acc_k1=round(m[0], 3), acc_k31=round(m[-1], 3)))
sweep = pd.DataFrame(rows); display(sweep)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].errorbar(sweep.extra_noise, sweep.cv_acc, yerr=sweep.cv_std, marker='o', capsize=3)
ax[0].axhline(0.5, ls='--', c='gray'); ax[0].set(xlabel='extra noise sigma', ylabel='best CV accuracy')
ax[1].plot(sweep.extra_noise, sweep.best_k, marker='o'); ax[1].set(xlabel='extra noise sigma', ylabel='best k')
[a.grid(alpha=.3) for a in ax]; plt.tight_layout(); plt.savefig('knn_noise_sweep.png', dpi=160); plt.show()